# `integrate_results.csv`

Connor Shrader / 09-21-2025

This file loads in and computes metrics on all of the clone results and saves to the file `all_clone_results.csv`.

In [1]:
import os
import pandas as pd
import numpy as np
import ast
import testis_model as tm
from tqdm import tqdm

from saving_loading import find_scenarios, load_all_scenarios, load_clone_names
from testis_model.metrics import init_time, last_time, max_time, freq_at, size, avg_freq_change, run_likelihood

In [2]:
scenarios = find_scenarios()
fish_data, clone_data = load_all_scenarios(scenarios)
clone_names = load_clone_names()

In [3]:
print(f'Found scenarios are: {scenarios}.')

Found scenarios are: ['n500_r15_y6', 'n500_r1_y6', 'n1500_r36_y6', 'n500_r15_y9', 'n1000_r1_y3', 'n500_r1_y9', 'n1500_r36_y9', 'n1000_r15_y6', 'n1000_r0_y3', 'n500_r4_y6', 'n500_r0_y9', 'n1000_r4_y3', 'n1000_r15_y9', 'n500_r0_y6', 'n500_r4_y9', 'n1000_r36_y3', 'n1500_r4_y9', 'n1500_r0_y6', 'real', 'n1500_r0_y9', 'n1500_r4_y6', 'n500_r36_y3', 'n1500_r1_y9', 'n1500_r15_y3', 'n1500_r1_y6', 'n1000_r1_y6', 'n1500_r36_y3', 'n500_r1_y3', 'n500_r15_y3', 'n1000_r1_y9', 'n500_r0_y3', 'n1000_r4_y6', 'n1000_r0_y9', 'n500_r4_y3', 'n1000_r0_y6', 'n1000_r4_y9', 'n1000_r15_y3', 'n1500_r4_y3', 'n1000_r36_y9', 'n1500_r0_y3', 'n1000_r36_y6', 'n1500_r15_y9', 'n1000_r2.5_y6', 'n1500_r1_y3', 'n500_r36_y9', 'n1500_r15_y6', 'n500_r36_y6'].


In [4]:
long_dfs = []
for scenario in scenarios:
    path = os.path.join('scenarios', scenario, f'{scenario}_long.csv')
    if os.path.isfile(path):
        long_dfs.append(pd.read_csv(path))

In [5]:
long_results = pd.concat(long_dfs).set_index(['scenario', 'fish', 'clone'])
print(f'Long results has length {len(long_results)}.')
long_results.head()

Long results has length 35436.


init_time  init_freq      r_opt    if_opt       l_max  \
scenario    fish clone                                                          
n500_r15_y6 0    0            5.0   0.555256  26.547272  0.551790 -212.780255   
                 1            5.0   0.311770  24.128502  0.316866 -157.553822   
                 2            5.0   0.132974  23.602596  0.144224 -211.285386   
            1    0            5.0   0.365678  25.003799  0.369282 -120.251278   
                 1            5.0   0.331536  22.722833  0.336038 -192.071656   

                            lower       upper   if_null       l_null  
scenario    fish clone                                                
n500_r15_y6 0    0      11.670349   70.666996  0.288180 -2317.466955  
                 1      10.354475   67.457306  0.144305 -3094.812299  
                 2      10.235373   63.934861  0.569150 -5935.367621  
            1    0       8.721297  104.718456  0.122192 -2597.410266  
                 1       9.610288   63.928778  0.183544 -1403.052886

In [9]:
# Specify order of columns
columns = ['scenario', 'fish', 'clone', 'init_time', 'init_freq', 'last_time',
           'max_time', 'max_freq', 'size', 'avg_freq_change', 'r_opt', 'if_opt',
           'l_max', 'lower', 'upper', 'if_null', 'l_null']

rows = []
for (scenario, fish, clone), data in tqdm(clone_data.items()):
    try:
        if (scenario, fish, clone) in long_results.index:
            row = long_results.loc[scenario, fish, clone].to_dict()
        else:
            row = {col: pd.NA for col in columns}
            it = init_time(data)
            if it is None: continue
            else: row['init_time'] = it

            row['init_freq'] = freq_at(data, row['init_time'])
    
        row['scenario'] = scenario
        row['fish'] = fish
        row['clone'] = clone
        
        row['last_time'] = last_time(data)
        row['max_time'] = max_time(data)
        row['max_freq'] = freq_at(data, row['max_time'])
        row['size'] = size(data)
            
        row['avg_freq_change'] = avg_freq_change(data)
        
        rows.append(row)

    except Exception as e:
        print(f'Error for scenario={scenario}, fish={fish}, clone={clone}')
        raise

100%|████████████████████████████████████████| 46306/46306 [01:08<00:00, 678.86it/s]


In [10]:
df = pd.DataFrame(rows, columns = columns)
df.to_csv('all_clone_results.csv', index = False)

In [11]:
num_na = len(df[df['r_opt'].isna()])
num_fail = len(df[df['l_max'] < -1e5])

print(f'{num_na} rows have r_opt == NA, and {num_fail} have l_max < -1e5 (indicating a failure to converge).')
print(f'These problematic rows are shown below.')

df.query('r_opt.isna() or l_max < -1e5')

24 rows have r_opt == NA, and 24 have l_max < -1e5 (indicating a failure to converge).
These problematic rows are shown below.


,scenario,fish,clone,init_time,init_freq,last_time,max_time,max_freq,size,avg_freq_change,r_opt,if_opt,l_max,lower,upper,if_null,l_null
428,n500_r15_y6,94,0,5.00,1.000000,26.00,26.00,1.000000,1.000000,0.0,NaN,1.000000,-1000000.0,0.0,NaN,1.000000,0.000000
13809,real,23,4,5.76,0.008018,5.76,5.76,0.008018,0.004720,NaN,NaN,0.008086,-1000000.0,0.0,NaN,0.003961,-8.372344
13920,real,43,9,5.76,0.013672,5.76,5.76,0.013672,0.009485,NaN,NaN,0.013623,-1000000.0,0.0,NaN,0.003961,-8.372344
13966,real,6,2,5.76,0.029870,5.76,5.76,0.029870,0.014171,NaN,NaN,0.029875,-1000000.0,0.0,NaN,0.010488,-22.512597
15909,n500_r36_y3,16,0,5.00,1.000000,26.00,26.00,1.000000,1.000000,0.0,NaN,1.000000,-1000000.0,0.0,NaN,1.000000,0.000000
15912,n500_r36_y3,18,0,5.00,1.000000,26.00,26.00,1.000000,1.000000,0.0,NaN,1.000000,-1000000.0,0.0,NaN,1.000000,0.000000
15933,n500_r36_y3,24,0,5.00,1.000000,26.00,26.00,1.000000,1.000000,0.0,NaN,1.000000,-1000000.0,0.0,NaN,1.000000,0.000000
16043,n500_r36_y3,63,0,5.00,1.000000,26.00,26.00,1.000000,1.000000,0.0,NaN,1.000000,-1000000.0,0.0,NaN,1.000000,0.000000
31421,n1000_r36_y6,55,0,5.00,1.000000,26.00,26.00,1.000000,1.000000,0.0,NaN,1.000000,-1000000.0,0.0,NaN,1.000000,0.000000
34212,n500_r36_y9,1,0,5.00,1.000000,26.00,26.00,1.000000,1.000000,0.0,NaN,1.000000,-1000000.0,0.0,NaN,1.000000,0.000000
